In [ ]:
import os, re, gc, json, math, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

RANDOM_SEEDS = [42, 2026, 2027]
N_SPLITS = 5
ACTIVE_SEED = 42
ACTIVE_FOLD = 0

# Notebook thử tìm dữ liệu ở các vị trí phổ biến.
DATA_CANDIDATES = [
    Path("data/en-nn-train.tsv"),
    Path("en-nn-train.tsv"),
    Path("/mnt/data/en-nn-train.tsv"),
]
OUTPUT_DIR = Path("outputs_en_nn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLS = ["ModAvg", "HeadAvg"]
STD_COLS = ["ModStd", "HeadStd"]
EXPECTED_COLUMNS = [
    "ContextID", "Compound", "Mod", "Head",
    "ModAvg", "ModStd", "HeadAvg", "HeadStd", "Context"
]

# Hyperparameters; điều chỉnh sau khi có baseline.
STATIC_EMBEDDING_NAME = "glove-wiki-gigaword-100"
FROZEN_BERT_NAME = "bert-base-uncased"
FROZEN_BERT_LAYER = 4       # hidden_states[4] = đầu ra Transformer layer thứ 4
FINETUNE_MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 2

print("Output directory:", OUTPUT_DIR.resolve())

## 6. Chia dữ liệu theo target: StratifiedGroupKFold

In [ ]:
def build_strata(df, n_bins=5):
    # Một nhãn phân tầng cho mỗi Compound, dựa trên trung bình ModAvg của Compound đó.
    type_mod_mean = df.groupby("Compound")["ModAvg"].mean()
    q = min(n_bins, type_mod_mean.nunique())
    bins = pd.qcut(type_mod_mean, q=q, labels=False, duplicates="drop")
    return df["Compound"].map(bins).astype(int)

def make_group_folds(data, seeds=RANDOM_SEEDS, n_splits=N_SPLITS):
    strata = build_strata(data)
    all_folds = {}
    for seed in seeds:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        seed_folds = []
        for fold, (train_idx, val_idx) in enumerate(
            splitter.split(data, y=strata, groups=data["Compound"])
        ):
            train_groups = set(data.iloc[train_idx]["Compound"])
            val_groups = set(data.iloc[val_idx]["Compound"])
            assert train_groups.isdisjoint(val_groups), "Target leakage detected!"
            seed_folds.append((train_idx, val_idx))
        all_folds[seed] = seed_folds
    return all_folds

folds = make_group_folds(df)
train_idx, val_idx = folds[ACTIVE_SEED][ACTIVE_FOLD]
train_df = df.iloc[train_idx].copy().reset_index(drop=True)
val_df = df.iloc[val_idx].copy().reset_index(drop=True)

print("Seed / fold:", ACTIVE_SEED, ACTIVE_FOLD)
print("Train rows / compounds:", len(train_df), train_df["Compound"].nunique())
print("Valid rows / compounds:", len(val_df), val_df["Compound"].nunique())
print("Overlap compounds:", len(set(train_df.Compound) & set(val_df.Compound)))

display(pd.DataFrame([
    {"split": "train", "rows": len(train_df), "compounds": train_df.Compound.nunique()},
    {"split": "validation", "rows": len(val_df), "compounds": val_df.Compound.nunique()},
]))

# Lưu manifest để tất cả mô hình dùng đúng cùng một split.
manifest = pd.DataFrame({"row_index": np.arange(len(df)), "Compound": df["Compound"], "ContextID": df["ContextID"]})
manifest["split"] = "train"
manifest.loc[val_idx, "split"] = "validation"
manifest["seed"] = ACTIVE_SEED
manifest["fold"] = ACTIVE_FOLD
manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)

In [ ]:
def safe_spearman(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) < 2 or np.unique(a).size < 2 or np.unique(b).size < 2:
        return np.nan
    return float(spearmanr(a, b).statistic)

def evaluate_one_target(data, pred, gold_col, target_col="Compound"):
    gold = data[gold_col].to_numpy(dtype=float)
    pred = np.asarray(pred, dtype=float)
    rmse = float(np.sqrt(mean_squared_error(gold, pred)))
    rho = safe_spearman(gold, pred)

    tmp = pd.DataFrame({target_col: data[target_col].values, "gold": gold, "pred": pred})
    grouped = tmp.groupby(target_col).agg(
        gold_std=("gold", lambda x: np.std(x, ddof=0)),
        pred_std=("pred", lambda x: np.std(x, ddof=0)),
        n=("gold", "size")
    )
    grouped = grouped[grouped["n"] >= 2].copy()
    # Chuẩn hóa sai số số học rất nhỏ thành 0; tránh Spearman giả khi prediction hằng số.
    grouped.loc[grouped["pred_std"] <= 1e-12, "pred_std"] = 0.0
    grouped.loc[grouped["gold_std"] <= 1e-12, "gold_std"] = 0.0
    context_var_rho = safe_spearman(grouped["gold_std"], grouped["pred_std"])
    zero_pred_std = int((grouped["pred_std"] <= 1e-12).sum())
    return {
        "Spearman": rho,
        "Context_variance_rho": context_var_rho,
        "RMSE": rmse,
        "n_compounds_for_variance": len(grouped),
        "zero_pred_std_compounds": zero_pred_std,
    }

def evaluate_predictions(data, pred_mod, pred_head, model_name="model"):
    rows = []
    for label, pred in zip(LABEL_COLS, [pred_mod, pred_head]):
        metrics = evaluate_one_target(data, pred, label)
        rows.append({"model": model_name, "target": label, **metrics})
    return pd.DataFrame(rows)

## 11. B3 – DeBERTa fine-tuning cho hai đầu ra

Mô hình nhận cặp đầu vào:

`[marked context] [SEP] [compound]`

Trong câu, modifier và head được đánh dấu bằng special tokens `<m>...</m>` và `<h>...</h>`. Encoder DeBERTa được fine-tune và một regression head dự đoán đồng thời `ModAvg`, `HeadAvg`. Đầu ra dùng `5 × sigmoid` để nằm trong khoảng 0–5.

Mặc định chỉ huấn luyện `ACTIVE_FOLD` để kiểm tra pipeline. Khi chạy báo cáo cuối, nên lặp đủ 5 folds và 3 seeds, lưu dự đoán out-of-fold.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import copy

SPECIAL_MARKERS = ["<m>", "</m>", "<h>", "</h>"]

def mark_context(row):
    s = row["Context"]
    span = row["_span"]
    if span is None:
        raise ValueError(f"Không tìm thấy compound span: {row['Compound']} | {s}")
    return (
        s[:span["mod_start"]] + "<m>" + s[span["mod_start"]:span["mod_end"]] + "</m>" +
        s[span["mod_end"]:span["head_start"]] +
        "<h>" + s[span["head_start"]:span["head_end"]] + "</h>" +
        s[span["head_end"]:]
    )

class CompoundDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=MAX_LENGTH):
        marked = [mark_context(r) for _, r in frame.iterrows()]
        compounds = frame["Compound"].tolist()
        enc = tokenizer(marked, compounds, truncation=True, max_length=max_length,
                        padding="max_length", return_tensors="pt")
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = torch.tensor(frame[LABEL_COLS].to_numpy(dtype=np.float32))
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

class DebertaRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.regressor = nn.Linear(hidden, 2)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return 5.0 * torch.sigmoid(self.regressor(self.dropout(cls)))

In [ ]:
def predict_deberta(model, loader):
    model.eval()
    preds, golds = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            preds.append(model(ids, mask).cpu().numpy())
            golds.append(batch["labels"].numpy())
    return np.vstack(preds), np.vstack(golds)

def run_deberta_fold(train_frame, valid_frame, fold_name="fold0", seed=ACTIVE_SEED):
    # Cố định seed để có thể tái lập lần chạy.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(FINETUNE_MODEL_NAME, use_fast=True)
    tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_MARKERS})

    train_ds = CompoundDataset(train_frame, tokenizer)
    valid_ds = CompoundDataset(valid_frame, tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = DebertaRegressor(FINETUNE_MODEL_NAME).to(DEVICE)
    model.encoder.resize_token_embeddings(len(tokenizer))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_score, best_state, best_epoch, bad_epochs = -np.inf, None, 0, 0
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for batch in train_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            preds = model(ids, mask)
            loss = F.mse_loss(preds, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())

        val_pred, val_gold = predict_deberta(model, valid_loader)
        val_metrics = evaluate_predictions(
            valid_frame, val_pred[:, 0], val_pred[:, 1], f"DeBERTa_epoch_{epoch}"
        )
        valid_rhos = val_metrics["Spearman"].dropna()
        score = float(valid_rhos.mean()) if len(valid_rhos) else -float(val_metrics["RMSE"].mean())
        history.append({"epoch": epoch, "train_mse": np.mean(losses), "selection_score": score,
                        "Mod_spearman": val_metrics.loc[0, "Spearman"],
                        "Head_spearman": val_metrics.loc[1, "Spearman"],
                        "Mod_RMSE": val_metrics.loc[0, "RMSE"],
                        "Head_RMSE": val_metrics.loc[1, "RMSE"]})
        print(f"Epoch {epoch}/{EPOCHS} | train MSE={np.mean(losses):.4f} | selection={score:.4f}")

        if score > best_score:
            best_score, best_epoch, bad_epochs = score, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    final_pred, final_gold = predict_deberta(model, valid_loader)
    final_metrics = evaluate_predictions(valid_frame, final_pred[:, 0], final_pred[:, 1], "B3_DeBERTa_finetuned")
    hist = pd.DataFrame(history)
    hist.to_csv(OUTPUT_DIR / f"B3_{fold_name}_history.csv", index=False)
    final_out = valid_frame[["ContextID", "Compound", "Context", "ModAvg", "HeadAvg"]].copy()
    final_out["pred_ModAvg"] = final_pred[:, 0]
    final_out["pred_HeadAvg"] = final_pred[:, 1]
    final_out.to_csv(OUTPUT_DIR / f"B3_{fold_name}_predictions.csv", index=False)
    model_path = OUTPUT_DIR / f"B3_{fold_name}_checkpoint"
    model.encoder.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    torch.save(model.regressor.state_dict(), model_path / "regression_head.pt")

    del model, train_loader, valid_loader, train_ds, valid_ds
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return final_metrics, hist, final_pred

In [ ]:
# Chạy DeBERTa fine-tuning. Có thể mất thời gian và cần tải model (~hàng trăm MB).
RUN_FINETUNING = False  # Đổi thành True khi sẵn sàng chạy.

if RUN_FINETUNING:
    metrics_deberta, history_deberta, pred_deberta = run_deberta_fold(
        train_df, val_df, fold_name=f"seed{ACTIVE_SEED}_fold{ACTIVE_FOLD}"
    )
    display(metrics_deberta)
    display(history_deberta)
else:
    print("Chưa chạy fine-tuning. Đổi RUN_FINETUNING=True để bắt đầu.")